# Ricci Finance v11.0 Lecture Notebook — colorbar + 3D Ricci-flow version

## Abstract

This notebook studies a financial market as a **dynamic Ricci-capital manifold**.  Each ticker is a node, rolling return correlation defines the geometric distance between tickers, Ollivier-Ricci curvature measures local network coherence or bridge stress, and dollar volume adds a capital-mass correction.  The purpose is not to claim that the market has reached a final Perelman-resolved state.  Instead, the goal is to build a rolling diagnostic that shows how clusters form, rotate, fragment, and transmit capital flow through time.

## Purpose

1. Build an IPO-aware rolling market network where late-start tickers remain visible before enough data exists.
2. Convert rolling correlations into financial distances.
3. Compute Ricci curvature to detect coherent basins and fragile bridges.
4. Add dollar-volume mass so the network reflects both geometry and capital transport.
5. Use 3D visualization to inspect the final manifold and the before/after Ricci-flow projection.
6. Fit an optional HMM on Ricci-capital features to identify hidden market regimes.

## Method summary

- **Data:** raw market Close and Volume from `yfinance`; fallback to synthetic lecture data if live data is unavailable.
- **Returns:** log returns from Close prices.
- **Distance:** Mantegna-style distance, `d_ij = sqrt(2(1 - rho_ij))`, optionally adjusted by dollar-volume similarity.
- **Graph:** `knn+bridges` keeps each ticker connected to nearest positive-correlation neighbors, then adds a few weak bridge edges for readable cross-theme structure.
- **Curvature:** Ollivier-Ricci curvature when `GraphRicciCurvature` is available; otherwise a didactic fallback keeps the lecture executable.
- **Ricci flow:** edge weights update by `w_{t+1} = w_t * (1 - step_size * kappa_t)`; positive curvature contracts, negative curvature stretches.
- **3D manifold:** x/y are stable topology, z is Ricci stress by default, node size is capital mass, edge width is capital transport, and edge color is curvature.

## What this notebook has done after all cells run

It loads market data, builds rolling Ricci-capital frames, displays a dynamic 3D manifold, explains each table column, compares selected-frame curvature and capital transport, runs Ricci flow, replaces the old 2D before/after flow picture with the defined 3D function, and summarizes hidden regimes.


## v11 visual encoding rule: edge color now has a fixed colorbar

**Subject:** make every network picture quantitatively readable.

**Method:** all 2D, 3D, animated, and before/after Ricci-flow figures use the same fixed Ricci-curvature scale: κ ∈ [-0.6, +0.6].

**Result interpretation:**

- **Red edge** = negative Ricci curvature: fragile bridge / stress-transmission route.
- **White edge** = near-zero curvature: neutral relation.
- **Blue edge** = positive Ricci curvature: coherent cluster / capital basin.
- **Edge thickness** = dollar-volume capital transport.
- **Node size** = dollar-volume market mass.
- **Node color** = cluster/component.
- **Gray node** = active universe member but waiting for enough data, such as pre-IPO periods.

Therefore the colors are **not all the same**; they encode curvature with a visible, fixed, lecture-safe colorbar.


## Cell 1 — Imports and lecture controls

**Subject:** load helper functions and define reproducible notebook controls.

**Expected result:** after this cell runs, all functions from `helper.py` are available, and the parameter dictionary records the tickers, rolling-window size, graph mode, Ricci-flow settings, and 3D z-axis mode used in the rest of the notebook.


In [ ]:
from helper import *
import pandas as pd
from IPython.display import display, Markdown

PARAMS = {
    "tickers": DEFAULT_TICKERS,
    "period": "1y",
    "interval": "1d",
    "window_size": 60,
    "step": 5,
    "max_frames": 30,
    "graph_mode": "knn+bridges",
    "k_neighbors": 3,
    "min_corr": 0.05,
    "max_bridges": 3,
    "capital_alpha": 0.35,
    "ricci_alpha": 0.5,
    "ricci_method": "OTD",
    "flow_iterations": 8,
    "flow_step_size": 0.25,
    "z_mode": "ricci_stress",
    "layout_seed": 42,
}

display(Markdown("### Result of Cell 1"))
display(pd.Series(PARAMS, name="value").to_frame())


## Cell 2 — Market data: Close, Volume, dollar volume, and returns

**Subject:** download real market data and compute log returns.

**Method:** use `download_market_data()` so `prices.tail()` reports raw Yahoo/market `Close` rather than adjusted prices.  If live download fails or has too little usable data, use IPO-aware synthetic data so the lecture still runs offline.

**Expected result:** four objects are created: `prices`, `volumes`, `dollar_volume`, and `returns`.  The displayed table shows recent Close prices, and the printed summary reports the number of rows, active tickers, and date range.


In [ ]:
tickers = PARAMS["tickers"]

try:
    prices, volumes, dollar_volume = download_market_data(
        tickers=tickers,
        period=PARAMS["period"],
        interval=PARAMS["interval"],
    )
    data_source = "yfinance live raw Close/Volume"
except Exception as exc:
    print(f"Live market download failed: {exc}")
    prices, volumes, dollar_volume = make_demo_market_data(tickers=tickers, n_days=260, seed=7)
    data_source = "synthetic IPO-aware lecture data"

returns = prices_to_returns(prices).dropna(axis=1, how="all")
dollar_volume = dollar_volume.reindex(index=returns.index, columns=returns.columns).fillna(0.0)

display(Markdown("### Result of Cell 2"))
print(f"Data source: {data_source}")
print(f"Close rows x tickers: {prices.shape}")
print(f"Return rows x tickers: {returns.shape}")
print(f"Date range: {returns.index.min()} → {returns.index.max()}")
display(prices.tail())


## Cell 3 — Build rolling Ricci-capital frames

**Subject:** convert rolling return windows into Ricci-capital graphs.

**Method:** each rolling window creates a graph.  Nodes are tickers; edges are selected by `knn+bridges`; edge distance comes from correlation and capital weighting; curvature is computed on each edge; dollar volume is stored as node mass and edge transport.

**Expected result:** `frames` is a list of rolling `FrameData` objects, `starts` records each window start index, and `feature_df` summarizes the market geometry through time.

### Rolling feature table column meanings

| Column | Meaning |
|---|---|
| `date` | End date of the rolling window. |
| `avg_ricci` | Mean edge Ricci curvature. Higher usually means more coherent clustered market geometry. |
| `ricci_std` | Dispersion of edge curvature. Higher means mixed coherent and bridge-stress edges. |
| `clusters` | Number of connected components. Higher means more fragmentation. |
| `largest_component_ratio` | Fraction of tickers in the largest connected component. |
| `nodes`, `edges`, `density` | Basic graph size and connection density. |
| `total_node_capital` | Sum of node dollar-volume masses in the frame. |
| `total_edge_capital_flow` | Sum of edge dollar-volume transport intensities. |
| `avg_edge_capital_flow` | Average transport per edge. |
| `max_node_capital_share` | Concentration of capital mass in the largest node. |


In [ ]:
frames, starts = build_rolling_frames(
    returns=returns,
    window_size=PARAMS["window_size"],
    step=PARAMS["step"],
    max_frames=PARAMS["max_frames"],
    graph_mode=PARAMS["graph_mode"],
    k_neighbors=PARAMS["k_neighbors"],
    min_corr=PARAMS["min_corr"],
    max_bridges=PARAMS["max_bridges"],
    max_distance=1.35,
    min_abs_corr=0.10,
    alpha=PARAMS["ricci_alpha"],
    method=PARAMS["ricci_method"],
    proc=1,
    min_node_obs=1,
    min_pair_obs=4,
    dollar_volume=dollar_volume,
    use_capital_weighting=True,
    capital_alpha=PARAMS["capital_alpha"],
)

feature_df = rolling_feature_table(frames)

display(Markdown("### Result of Cell 3"))
print(f"Built {len(frames)} rolling frames.")
display(feature_df.head())
display(plot_rolling_features(feature_df))


## Cell 4 — Dynamic 3D Ricci-capital manifold

**Subject:** visualize rolling market geometry as an interactive 3D animation.

**Method:** use the defined `compute_stable_layout_3d()` and `build_3d_ricci_capital_animation()` functions.

**Expected result:** an interactive Plotly 3D figure appears.  Use mouse rotation to inspect clusters and bridges.  Press Play/Pause to move through rolling windows.

### Visual encoding

| Visual element | Meaning |
|---|---|
| x/y position | Stable network topology built from all frames. |
| z-axis | Ricci stress by default; high z means more negative-curvature bridge stress. |
| node size | Dollar-volume capital mass. |
| node color | Connected component / cluster. |
| edge color | Ricci curvature. Positive/coherent and negative/bridge-like edges are visually separated. |
| edge width | Dollar-volume capital transport intensity. |
| animation frame | Rolling-window time evolution. |


In [ ]:
base_graph = build_base_graph_for_layout(frames, all_nodes=returns.columns)
positions_3d = compute_stable_layout_3d(base_graph, seed=PARAMS["layout_seed"], layout_k=0.45)

fig3d = build_3d_ricci_capital_animation(
    frames=frames,
    positions_3d=positions_3d,
    z_mode=PARAMS["z_mode"],
    frame_duration_ms=700,
    edge_label_top_n=25,
)

display(Markdown("### Result of Cell 4"))
display(Markdown("The 3D manifold is ready. Rotate it to inspect bridges, compact clusters, and capital-mass concentration."))
display(fig3d)


## Cell 5 — Inspect one rolling frame and explain the edge table

**Subject:** choose one frame, draw its 3D manifold, and inspect edge-level geometry.

**Method:** select the last rolling frame by default.  Use `visualize_network_3d()` for the selected frame and `summarize_edges()` for the edge table.

**Expected result:** a static 3D graph for the chosen frame and an edge table sorted by shortest effective distance.

### Edge table: distance, correlation, curvature, and capital flow

| Column | Meaning |
|---|---|
| `u`, `v` | The two tickers connected by the edge. |
| `distance` | Effective financial distance used by the graph. Smaller means the pair is closer after correlation and optional capital weighting. |
| `raw_distance` | Pure correlation distance before capital correction. Compare with `distance` to see the capital-weighting effect. |
| `correlation` | Rolling-window return correlation. Larger positive value means more synchronized returns. |
| `ricciCurvature` | Ollivier-Ricci curvature. Positive often indicates a coherent/redundant local basin; negative often indicates a bridge-like stress channel. |
| `edge_capital_flow` | Dollar-volume weighted transport intensity through the pair. Larger means more capital is associated with that relation. |
| `capital_similarity` | Similarity of the two nodes' capital masses. Higher similarity strengthens the capital-weighting relation. |
| `edge_source` | Construction source: kNN, bridge, threshold, or related graph-building label. |


In [ ]:
inspect_idx = len(frames) - 1
fd = frames[inspect_idx]
edge_df = summarize_edges(fd.G)

selected_fig_3d = visualize_network_3d(
    fd.G,
    positions_3d=positions_3d,
    title=f"Selected frame {inspect_idx + 1}: 3D Ricci-capital manifold ({fd.stats.end_date})",
    node_cluster=fd.node_cluster,
    z_mode=PARAMS["z_mode"],
)

display(Markdown("### Result of Cell 5"))
print(f"Selected frame: {inspect_idx + 1}/{len(frames)}; end date = {fd.stats.end_date}")
print(f"Nodes={fd.stats.num_nodes}, Edges={fd.stats.num_edges}, Clusters={fd.stats.num_clusters}, Avg Ricci={fd.stats.avg_ricci:.4f}")
display(selected_fig_3d)
display(edge_df.head(20))


## Cell 6 — Capital-flow transport among clusters

**Subject:** measure where the market's dollar-volume mass sits and which edges carry the strongest transport.

**Method:** use node, edge, and cluster capital tables.  The edge flow table ranks the largest dollar-volume transport channels; the cluster table identifies capital basins.

**Expected result:** bar chart of the largest capital-flow edges plus three tables for edge transport, node market mass, and cluster capital basins.

### Capital-flow table column meanings

| Column | Meaning |
|---|---|
| `edge_capital_flow` | Dollar-volume weighted flow strength across one edge. |
| `edge_flow_share` | Share of total edge transport carried by that edge. |
| `capital_similarity` | How similar the two endpoint nodes are in dollar-volume mass. |
| `effective_distance` | Final distance used by graph construction after capital correction. |
| `raw_corr_distance` | Original correlation-only distance. |
| `correlation`, `ricciCurvature` | Return synchronization and local geometric stress/coherence. |

### Node/cluster capital meanings

`capital_mass` is the node's dollar-volume mass.  `capital_share` is its share of total frame mass.  A cluster with high `capital_share` is a capital basin.


In [ ]:
flow_df = capital_flow_table(fd.G)
node_cap_df = node_capital_table(fd.G)
cluster_cap_df = cluster_capital_table(fd.G, fd.node_cluster)

fig_flow = plot_capital_flow_bars(flow_df)

display(Markdown("### Result of Cell 6"))
display(fig_flow)
display(Markdown("**Top transport edges**"))
display(flow_df.head(20))
display(Markdown("**Node market mass**"))
display(node_cap_df.head(20))
display(Markdown("**Cluster capital basins**"))
display(cluster_cap_df)


## Cell 7 — Surgery-risk direction, not actual surgery

**Subject:** rank potential fragile bridge directions without cutting the graph.

**Method:** `surgery_risk_direction_table()` combines curvature, edge distance, and bridge-like structure to identify edges that may stretch, rotate, or become separation channels if stress increases.

**Expected result:** a ranked table.  Read this as **directional risk**, not as a command to delete edges.  Real markets are almost always in transition, not in a final resolved manifold.


In [ ]:
surgery_df = surgery_risk_direction_table(fd.G)

display(Markdown("### Result of Cell 7"))
display(surgery_df.head(30))


## Cell 8 — Ricci flow projection with 3D before/after views

**Subject:** project the selected frame through discrete Ricci flow and inspect the deformation.

**Method:** use `run_ricci_flow()` to update edge weights by curvature.  Then use the already-defined `visualize_network_3d()` function to replace the old 2D before/after comparison.

**Expected result:** a flow-history chart, a 3D graph before flow, a 3D graph after flow, and a comparison table.

### How to read Ricci flow

- Positive curvature edge: contracts under flow; it behaves like a coherent basin link.
- Negative curvature edge: stretches under flow; it behaves like a fragile bridge or stress-transmission channel.
- `delta_weight < 0`: edge contracted after flow.
- `delta_weight > 0`: edge stretched after flow.


In [ ]:
flowed_G, flow_history = run_ricci_flow(
    fd.G,
    iterations=PARAMS["flow_iterations"],
    step_size=PARAMS["flow_step_size"],
    alpha=PARAMS["ricci_alpha"],
    method=PARAMS["ricci_method"],
    proc=1,
    normalize_mean_weight=True,
)

before_flow_3d = visualize_network_3d(
    fd.G,
    positions_3d=positions_3d,
    title="Before Ricci flow — 3D Ricci-capital manifold",
    node_cluster=fd.node_cluster,
    z_mode=PARAMS["z_mode"],
)

after_flow_3d = visualize_network_3d(
    flowed_G,
    positions_3d=positions_3d,
    title="After Ricci flow — 3D Ricci-capital manifold",
    node_cluster=compute_components(flowed_G),
    z_mode=PARAMS["z_mode"],
)

comparison = compare_before_after_flow(fd.G, flowed_G)

display(Markdown("### Result of Cell 8"))
display(plot_ricci_flow_history(flow_history))
display(Markdown("**Before flow — 3D view**"))
display(before_flow_3d)
display(Markdown("**After flow — 3D view**"))
display(after_flow_3d)
display(comparison.head(30))


## Cell 9 — HMM hidden regimes from Ricci + capital features

**Subject:** infer unsupervised hidden market regimes from the rolling Ricci-capital feature table.

**Method:** fit a Gaussian HMM to curvature, density, cluster structure, and capital-flow features.  Regime labels are not supervised truth; they are names assigned by feature diagnostics.

**Expected result:** `hmm_df` adds a hidden-state number and regime name to each rolling frame; `regime_summary` summarizes each state.


In [ ]:
hmm_df, regime_names = compute_hmm_regimes(
    frames=frames,
    returns=returns,
    starts=starts,
    window_size=PARAMS["window_size"],
    n_components=3,
    forward_days=5,
    random_state=42,
)

fwd_col = "next_5d_market_return"
agg = {
    "avg_ricci": "mean",
    "density": "mean",
    "largest_component_ratio": "mean",
    "date": "count",
}
if fwd_col in hmm_df.columns:
    agg[fwd_col] = "mean"
regime_summary = hmm_df.groupby(["hmm_state", "regime_name"]).agg(agg).rename(columns={"date": "count"}).reset_index()

display(Markdown("### Result of Cell 9"))
display(plot_hmm_regimes(hmm_df))
display(hmm_df.tail())
display(Markdown("**Regime summary**"))
display(regime_summary)


## Final interpretation checklist

After running the full notebook, interpret the result in this order:

1. **Data validity:** confirm recent Close prices and enough return rows exist.
2. **Rolling geometry:** use `avg_ricci`, `density`, and `clusters` to judge whether the market is coherent, rotating, or fragmenting.
3. **3D manifold:** rotate the graph and inspect high-z stress bridges, large capital-mass nodes, and capital-flow edge width.
4. **Edge table:** compare `distance` vs `raw_distance`; if `distance` is much smaller than `raw_distance`, capital weighting pulled the pair closer.
5. **Capital basins:** use cluster capital share to identify where dollar volume is concentrated.
6. **Surgery-risk direction:** rank future separation candidates without cutting the graph.
7. **Ricci flow:** inspect 3D before/after views; stretched edges are possible stress channels, contracted edges are coherent basin links.
8. **HMM regimes:** use hidden states as a compact summary, but validate them against the visible Ricci-capital geometry.
